In [2]:
"""
GPU Post-Training Quantization (PTQ) — ResNet-50 / CIFAR-10
=============================================================
GPU-native PTQ using multiple frameworks, since PyTorch's static PTQ
is CPU-only. This script benchmarks three GPU-capable approaches:

  1. ONNX Runtime  — export FP32 → ONNX → INT8 via ORT quantization API
                     (CUDAExecutionProvider for inference)
  2. TensorRT      — export FP32 → ONNX → TRT INT8 engine with calibration
                     (requires tensorrt + pycuda; best GPU throughput)
  3. Torch CUDA    — dynamic PTQ (Linear → qint8) run on CUDA tensors;
                     static PTQ via torch.ao on CUDA (experimental, requires nightly)

FLOPs measurement:
  - fvcore.nn.FlopCountAnalysis for FP32/quantized PyTorch models
  - Manual formula for conv/linear: FLOPs = 2 × Cin × Cout × Kh × Kw × H × W
  - For INT8: effective FLOPs halved (wider SIMD lanes, fused multiply-add)

Mathematical foundation (same as CPU PTQ, different execution backend):
  Q(x)  = round(x / S) + Z              [quantization]
  x̂     = S · (Q(x) − Z)               [dequantization]
  ε     = x − x̂,  |ε| ≤ S/2           [bounded error]
  S     = (x_max − x_min) / (2^b − 1)  [scale]
  Z     = round(−x_min / S)             [zero-point]

GPU throughput advantage:
  INT8 GEMM on Tensor Cores: ~4× vs FP32 on Ampere+
  TensorRT fuses layers, eliminates memory round-trips
  ONNX Runtime CUDAExecutionProvider uses cuDNN INT8 kernels

Output: gpu_ptq_results.json
"""

# ── Imports ────────────────────────────────────────────────────────────────────
import os
import sys
import json
import time
import copy
import math
import tempfile
import warnings
import argparse
import subprocess
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np

warnings.filterwarnings("ignore")

# ── Config ─────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "gpu_ptq_results_v2.json"

BATCH_SIZE     = 128
NUM_WORKERS    = 4
NUM_CLASSES    = 10
CALIB_SIZE     = 1024
CALIB_BATCHES  = 8
INFERENCE_RUNS = 50
INPUT_SHAPE    = (1, 3, 32, 32)   # single-image shape for export/FLOPs

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

# ── Dependency detection ───────────────────────────────────────────────────────
def _try_import(name: str):
    try:
        import importlib
        return importlib.import_module(name)
    except ImportError:
        return None

torch         = _try_import("torch")
torchvision   = _try_import("torchvision")
ort           = _try_import("onnxruntime")
onnx          = _try_import("onnx")
onnxquant     = _try_import("onnxruntime.quantization")
fvcore        = _try_import("fvcore")
tensorrt      = _try_import("tensorrt")
pycuda        = _try_import("pycuda")
bitsandbytes  = _try_import("bitsandbytes")

# ── Install helpers (run once if needed) ───────────────────────────────────────
INSTALL_COMMANDS = {
    "torch":          "pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121",
    "onnx":           "pip install onnx onnxruntime-gpu",
    "onnxruntime":    "pip install onnxruntime-gpu",  # GPU build
    "fvcore":         "pip install fvcore",
    "tensorrt":       "pip install tensorrt",         # NVIDIA wheel
    "bitsandbytes":   "pip install bitsandbytes",
    "optimum":        "pip install optimum[onnxruntime-gpu]",
}

def install_deps():
    """Install all GPU PTQ dependencies. Run once before the experiment."""
    cmds = [
        # PyTorch with CUDA 12.1
        "pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121",
        # ONNX ecosystem (GPU runtime)
        "pip install onnx onnxruntime-gpu",
        # FLOPs counter
        "pip install fvcore",
        # TensorRT Python bindings (NVIDIA GPU required)
        "pip install tensorrt pycuda",
        # BitsAndBytes INT8/NF4 GPU quantization
        "pip install bitsandbytes",
        # Optimum for HuggingFace-style ONNX export + quantization
        "pip install optimum[onnxruntime-gpu]",
    ]
    for cmd in cmds:
        print(f"  $ {cmd}")
        subprocess.run(cmd.split(), check=False)


# ══════════════════════════════════════════════════════════════════════════════
# Model & Data
# ══════════════════════════════════════════════════════════════════════════════
def build_model(num_classes: int = 10):
    """ResNet-50 adapted for CIFAR-10 (32×32 input)."""
    import torch.nn as nn
    from torchvision import models
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path: str):
    model = build_model(NUM_CLASSES)
    model.load_state_dict(torch.load(path, map_location="cpu", weights_only=True))
    model.eval()
    return model

def get_test_loader():
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = datasets.CIFAR10(root="../data", train=False, download=True, transform=tf)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)

def get_calib_loader():
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader, Subset
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds  = datasets.CIFAR10(root="../data", train=True, download=True, transform=tf)
    sub = Subset(ds, list(range(CALIB_SIZE)))
    return DataLoader(sub, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)


# ══════════════════════════════════════════════════════════════════════════════
# FLOPs Measurement
# ══════════════════════════════════════════════════════════════════════════════
def count_flops_fvcore(model, input_shape=INPUT_SHAPE) -> Dict:
    """
    Use fvcore for accurate FLOPs counting.
    fvcore counts multiply-accumulate ops (MACs); we report GFLOPs = 2×MACs / 1e9.

    For a Conv2d layer:
      MACs = Cout × Cin × Kh × Kw × Hout × Wout
      FLOPs = 2 × MACs  (one multiply + one accumulate per MAC)

    For a Linear layer:
      MACs = Cout × Cin
      FLOPs = 2 × MACs
    """
    from fvcore.nn import FlopCountAnalysis, parameter_count
    x = torch.randn(*input_shape)
    flops = FlopCountAnalysis(model, x)
    flops.unsupported_ops_warnings(False)
    total_macs = flops.total()
    params     = parameter_count(model)[""]
    return {
        "total_macs"    : int(total_macs),
        "total_flops"   : int(total_macs * 2),   # FLOPs = 2 × MACs
        "gflops_fp32"   : round(total_macs * 2 / 1e9, 4),
        "gflops_int8_theoretical": round(total_macs * 2 / 1e9 / 4, 4),  # ~4× speedup
        "params_total"  : int(params),
        "params_M"      : round(params / 1e6, 3),
        "note": (
            "FLOPs = 2 × MACs (one multiply + one accumulate). "
            "INT8 theoretical is FP32/4 because INT8 Tensor Core throughput "
            "is 4× FP32 on Ampere+ (A100, RTX 30xx+)."
        ),
    }

def count_flops_manual(model) -> Dict:
    """
    Manual FLOPs calculation by walking named modules.
    Formula per layer:
      Conv2d  : FLOPs = 2 × Cin × Cout × Kh × Kw × Hout × Wout × N
      Linear  : FLOPs = 2 × in_features × out_features × N
      BN      : FLOPs = 4 × C × H × W × N  (normalize, scale, shift, var)

    H_out = floor((H_in + 2p - Kh) / stride + 1)
    """
    import torch.nn as nn

    # Forward hook to capture spatial dims
    hooks, spatial = [], {}

    def make_hook(name):
        def hook(module, inp, out):
            if hasattr(out, "shape"):
                spatial[name] = tuple(out.shape)
        return hook

    handles = []
    for name, mod in model.named_modules():
        handles.append(mod.register_forward_hook(make_hook(name)))

    x = torch.randn(*INPUT_SHAPE)
    with torch.no_grad():
        model(x)
    for h in handles:
        h.remove()

    total_flops = 0
    layer_breakdown = {}

    for name, mod in model.named_modules():
        if isinstance(mod, nn.Conv2d):
            shape = spatial.get(name)
            if shape is None:
                continue
            _, cout, hout, wout = shape
            cin  = mod.in_channels
            kh   = mod.kernel_size[0] if isinstance(mod.kernel_size, tuple) else mod.kernel_size
            kw   = mod.kernel_size[1] if isinstance(mod.kernel_size, tuple) else mod.kernel_size
            # MACs = Cout × (Cin/groups) × Kh × Kw × Hout × Wout
            macs = cout * (cin // mod.groups) * kh * kw * hout * wout
            flops = 2 * macs
            total_flops += flops
            layer_breakdown[name] = {"type": "Conv2d", "flops": flops, "macs": macs,
                                      "shape_out": list(shape)}

        elif isinstance(mod, nn.Linear):
            flops = 2 * mod.in_features * mod.out_features
            total_flops += flops
            layer_breakdown[name] = {"type": "Linear", "flops": flops,
                                      "in": mod.in_features, "out": mod.out_features}

        elif isinstance(mod, nn.BatchNorm2d):
            shape = spatial.get(name)
            if shape:
                _, c, h, w = shape
                flops = 4 * c * h * w   # normalize + scale + shift + var
                total_flops += flops
                layer_breakdown[name] = {"type": "BatchNorm2d", "flops": flops}

    return {
        "total_flops"    : total_flops,
        "gflops"         : round(total_flops / 1e9, 4),
        "layer_breakdown": layer_breakdown,
        "formula": {
            "conv2d":  "FLOPs = 2 × (Cin/groups) × Cout × Kh × Kw × Hout × Wout",
            "linear":  "FLOPs = 2 × in_features × out_features",
            "batchnorm": "FLOPs = 4 × C × H × W  (norm + scale + shift + var)",
        }
    }

def measure_gpu_throughput_ms(model, device, n: int = INFERENCE_RUNS) -> Dict:
    """
    GPU latency using CUDA events (more accurate than time.perf_counter).
    CUDA events are synchronized on the GPU timeline; avoids CPU-GPU sync overhead.

    Returns mean, std, min, max in milliseconds.
    """
    model = model.to(device).eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32, device=device)

    # Warmup
    with torch.no_grad():
        for _ in range(10):
            model(dummy)
    torch.cuda.synchronize(device)

    timings = []
    start_event = torch.cuda.Event(enable_timing=True)
    end_event   = torch.cuda.Event(enable_timing=True)

    with torch.no_grad():
        for _ in range(n):
            start_event.record()
            model(dummy)
            end_event.record()
            torch.cuda.synchronize()
            timings.append(start_event.elapsed_time(end_event))

    arr = np.array(timings)
    return {
        "mean_ms"  : round(float(arr.mean()), 4),
        "std_ms"   : round(float(arr.std()),  4),
        "min_ms"   : round(float(arr.min()),  4),
        "max_ms"   : round(float(arr.max()),  4),
        "samples"  : n,
        "batch_size": BATCH_SIZE,
        "throughput_imgs_per_sec": round(BATCH_SIZE / (float(arr.mean()) / 1000), 1),
    }

def measure_cpu_ms(model, n: int = INFERENCE_RUNS) -> float:
    model = model.cpu().eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32)
    with torch.no_grad():
        for _ in range(5):
            model(dummy)
        t0 = time.perf_counter()
        for _ in range(n):
            model(dummy)
    return round((time.perf_counter() - t0) / n * 1000, 4)

def disk_size_mb(obj) -> float:
    """Works for PyTorch models and ONNX files (pass path as str)."""
    if isinstance(obj, str):
        return os.path.getsize(obj) / 1024 ** 2
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(obj, tmp)
        return os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)


# ══════════════════════════════════════════════════════════════════════════════
# Evaluation
# ══════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def evaluate_torch(model, loader, device) -> Dict:
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    model = model.to(device).eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(device)
        preds.extend(model(inputs).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def evaluate_onnx(session, loader) -> Dict:
    """Run inference with ONNX Runtime session."""
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    input_name = session.get_inputs()[0].name
    preds, labels = [], []
    for inputs, lbls in loader:
        out = session.run(None, {input_name: inputs.numpy()})[0]
        preds.extend(np.argmax(out, axis=1))
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


# ══════════════════════════════════════════════════════════════════════════════
# ONNX Export
# ══════════════════════════════════════════════════════════════════════════════
def export_to_onnx(model, output_path: str, opset: int = 17) -> str:
    """
    Export PyTorch FP32 model to ONNX.
    opset 17+ is required for INT8 quantization ops (QLinearConv, QLinearMatMul).
    Dynamic axes allow variable batch size at inference time.
    """
    dummy = torch.randn(*INPUT_SHAPE)
    torch.onnx.export(
        model.cpu().eval(),
        dummy,
        output_path,
        opset_version        = opset,
        input_names          = ["input"],
        output_names         = ["output"],
        dynamic_axes         = {"input": {0: "batch_size"}, "output": {0: "batch_size"}},
        do_constant_folding  = True,
        export_params        = True,
    )
    print(f"        ✓ ONNX exported → {output_path}  "
          f"({os.path.getsize(output_path)/1024**2:.2f} MB, opset={opset})")
    return output_path


# ══════════════════════════════════════════════════════════════════════════════
# 1. ONNX Runtime — INT8 Static Quantization (GPU inference)
# ══════════════════════════════════════════════════════════════════════════════
"""
ONNX Runtime quantization workflow:
  1. Export FP32 model to ONNX (float32 weights + ops)
  2. Insert QuantizeLinear / DequantizeLinear nodes around each op
  3. Run calibration data through — ORT records tensor statistics
  4. Compute S = (x_max - x_min) / 255 and Z = round(-x_min / S) per tensor
  5. Freeze S and Z into the graph → INT8 model
  6. At inference: CUDAExecutionProvider maps QLinearConv → cuDNN INT8 kernel

Operators quantized by default:
  Conv, MatMul, Gemm, Relu, Add (for residual), GlobalAveragePool

Calibration strategies in ORT:
  MinMax      — exact range, sensitive to outliers (S too large → coarse grid)
  Entropy     — minimizes KL divergence between FP32 and INT8 distributions
  Percentile  — clips at [p, 1-p] to ignore outliers, reduces S → finer grid
"""
def run_onnx_ptq(fp32_model, test_loader, calib_loader,
                 baseline_acc: float, base_disk: float,
                 device: str, flops_fp32: Dict) -> List[Dict]:
    import onnxruntime
    from onnxruntime.quantization import (
        quantize_static, CalibrationDataReader,
        QuantType, QuantFormat, CalibrationMethod
    )

    print("\n  [1/3] ONNX Runtime — Static INT8 PTQ")
    print("        Framework: onnxruntime-gpu + cuDNN INT8 kernels")
    print("        Ops:       QLinearConv, QLinearMatMul, QLinearAdd")
    print("        S/Z:       calibrated once from calib data; frozen in graph")

    # Export FP32 → ONNX
    fp32_onnx = "resnet50_fp32.onnx"
    export_to_onnx(fp32_model, fp32_onnx)

    # Calibration data reader
    class CIFARCalibReader(CalibrationDataReader):
        """
        Feeds CIFAR-10 images to the ONNX calibrator.
        Observer collects x_min, x_max per tensor.
        S = (x_max - x_min) / 255,  Z = round(-x_min / S)
        """
        def __init__(self, loader, max_batches=CALIB_BATCHES):
            self.data   = iter(loader)
            self.batches = 0
            self.max    = max_batches

        def get_next(self):
            if self.batches >= self.max:
                return None
            try:
                imgs, _ = next(self.data)
                self.batches += 1
                return {"input": imgs.numpy()}
            except StopIteration:
                return None

    rows = []
    calibration_methods = {
        "MinMax"     : CalibrationMethod.MinMax,
        "Entropy"    : CalibrationMethod.Entropy,
        "Percentile" : CalibrationMethod.Percentile,
    }

    for calib_name, calib_method in calibration_methods.items():
        print(f"        Calibration: {calib_name:12s}", end="", flush=True)
        int8_onnx = f"resnet50_int8_{calib_name.lower()}.onnx"
        try:
            # Quantize: inserts QuantizeLinear + DequantizeLinear nodes
            quantize_static(
                model_input          = fp32_onnx,
                model_output         = int8_onnx,
                calibration_data_reader = CIFARCalibReader(calib_loader),
                quant_format         = QuantFormat.QDQ,       # QuantizeDequantize pairs
                per_channel          = True,                   # per-channel weights
                reduce_range         = False,
                weight_type          = QuantType.QInt8,
                activation_type      = QuantType.QUInt8,
                calibrate_method     = calib_method,
                extra_options        = {
                    "EnableSubgraph"              : True,
                    "MatMulConstBOnly"            : False,
                    "AddQDQPairToWeight"          : True,
                    "QuantizeAllFixedZeroPointTensors": True,
                },
            )

            # Inference session — prefer CUDA, fall back to CPU
            providers = (["CUDAExecutionProvider", "CPUExecutionProvider"]
                         if "CUDAExecutionProvider" in onnxruntime.get_available_providers()
                         else ["CPUExecutionProvider"])
            sess_opts = onnxruntime.SessionOptions()
            sess_opts.graph_optimization_level = (
                onnxruntime.GraphOptimizationLevel.ORT_ENABLE_ALL
            )
            session = onnxruntime.InferenceSession(int8_onnx, sess_opts,
                                                    providers=providers)
            active_provider = session.get_providers()[0]

            metrics  = evaluate_onnx(session, test_loader)
            disk_mb  = disk_size_mb(int8_onnx)

            # ORT latency (wall-clock; for true GPU use CUDA events in TRT section)
            dummy_np = np.random.randn(BATCH_SIZE, 3, 32, 32).astype(np.float32)
            input_name = session.get_inputs()[0].name
            with torch.no_grad():
                for _ in range(5):
                    session.run(None, {input_name: dummy_np})
            t0 = time.perf_counter()
            for _ in range(INFERENCE_RUNS):
                session.run(None, {input_name: dummy_np})
            latency_ms = (time.perf_counter() - t0) / INFERENCE_RUNS * 1000

            # INT8 FLOPs: same computation, but INT8 Tensor Cores do 4× throughput
            # Effective FLOPs count stays same; "compute efficiency" is the gain
            flops_int8 = {
                "total_flops"           : flops_fp32.get("total_flops"),
                "gflops_fp32_equivalent": flops_fp32.get("gflops_fp32"),
                "gflops_int8_theoretical": flops_fp32.get("gflops_int8_theoretical"),
                "note": (
                    "ONNX INT8 executes QLinearConv via cuDNN INT8 GEMM. "
                    "Op count is the same as FP32; throughput gain comes from "
                    "wider SIMD (4× INT8 ops per cycle vs FP32 on Tensor Cores)."
                ),
            }

            row = {
                "backend"            : "onnxruntime_int8",
                "calibration_method" : calib_name,
                "execution_provider" : active_provider,
                "description"        : (
                    f"ONNX Runtime static INT8 PTQ (calibration={calib_name}). "
                    "QDQ format: QuantizeLinear/DequantizeLinear pairs wrap each op. "
                    f"S = (x_max − x_min) / 255, Z = round(−x_min / S). "
                    f"Calibrated on {CALIB_SIZE} CIFAR-10 training images. "
                    "GPU inference via CUDAExecutionProvider → cuDNN INT8 kernels."
                ),
                "quantization_math"  : {
                    "S"       : "S = (x_max − x_min) / (2^8 − 1)",
                    "Z"       : "Z = round(−x_min / S)",
                    "Q(x)"    : "Q(x) = round(x / S) + Z  → stored as INT8/UINT8",
                    "x̂"       : "x̂ = S · (Q(x) − Z)  → reconstructed at fusion boundary",
                    "ε_bound" : "|ε| ≤ S/2  (smaller S ↔ denser range ↔ less error)",
                    "format"  : "QDQ (QuantizeDequantize pairs, ONNX standard)",
                },
                "ops_quantized"      : "Conv, MatMul, Gemm, Add (residual), GlobalAveragePool",
                "weight_dtype"       : "INT8 (per-channel)",
                "activation_dtype"   : "UINT8 (per-tensor)",
                "calibration_samples": CALIB_SIZE,
                "accuracy"           : round(metrics["accuracy"],  6),
                "precision"          : round(metrics["precision"], 6),
                "recall"             : round(metrics["recall"],    6),
                "f1"                 : round(metrics["f1"],        6),
                "accuracy_drop"      : round(baseline_acc - metrics["accuracy"], 6),
                "size_disk_mb"       : round(disk_mb, 4),
                "disk_saved_mb"      : round(base_disk - disk_mb, 4),
                "compression_ratio"  : round(base_disk / disk_mb, 4) if disk_mb else None,
                "inference_latency"  : {"mean_ms": round(latency_ms, 4),
                                         "batch_size": BATCH_SIZE},
                "flops"              : flops_int8,
            }
            print(f" → Acc: {metrics['accuracy']:.4f}  "
                  f"Drop: {baseline_acc - metrics['accuracy']:+.4f}  "
                  f"Disk: {disk_mb:.2f} MB  Lat: {latency_ms:.1f} ms ({active_provider})")
            rows.append(row)

        except Exception as e:
            print(f" → FAILED: {e}")
            rows.append({
                "backend": "onnxruntime_int8",
                "calibration_method": calib_name,
                "description": f"FAILED: {e}",
                "accuracy": None,
            })

    # Cleanup exported ONNX files
    for f in [fp32_onnx] + [f"resnet50_int8_{m.lower()}.onnx"
                              for m in calibration_methods]:
        if os.path.exists(f):
            os.remove(f)

    return rows


# ══════════════════════════════════════════════════════════════════════════════
# 2. TensorRT — INT8 PTQ with Calibration Cache
# ══════════════════════════════════════════════════════════════════════════════
"""
TensorRT INT8 PTQ workflow:
  1. Parse ONNX model into TRT NetworkDefinition
  2. Set INT8 mode + attach IInt8EntropyCalibrator2
  3. Build engine: TRT fuses layers, selects fastest INT8 kernels per GPU
  4. Serialize engine to disk (.trt) — device-specific!

TensorRT advantages over ORT INT8:
  - Kernel autotuning at build time (profile-guided)
  - Layer fusion: Conv + BN + ReLU → single kernel call
  - Memory layout optimization (channel-last for Tensor Core alignment)
  - ~2-5× faster than ORT GPU on A100/H100 for ResNet-class models

Calibration: IInt8EntropyCalibrator2
  Minimizes KL divergence between FP32 and INT8 activation distributions.
  Better than MinMax for activations with long tails (common in ResNets).

  S = argmin_{S} KL( P_fp32(x) || P_int8(Q(x,S)) )
"""
def run_tensorrt_ptq(fp32_model, test_loader, calib_loader,
                     baseline_acc: float, base_disk: float,
                     device: str, flops_fp32: Dict) -> Dict:
    print("\n  [2/3] TensorRT — INT8 PTQ")
    try:
        import tensorrt as trt
        import pycuda.driver as cuda
        import pycuda.autoinit
    except ImportError as e:
        print(f"        ⚠ TensorRT/pycuda not available: {e}")
        print("          Install: pip install tensorrt pycuda")
        return {
            "backend"    : "tensorrt_int8",
            "description": f"SKIPPED — TensorRT not installed: {e}",
            "accuracy"   : None,
            "install"    : "pip install tensorrt pycuda",
        }

    print("        Framework: TensorRT INT8 engine")
    print("        Fusion:    Conv-BN-ReLU automatic, memory layout optimized")
    print("        Calib:     IInt8EntropyCalibrator2 (KL-divergence minimization)")

    TRT_LOGGER = trt.Logger(trt.Logger.WARNING)
    fp32_onnx  = "resnet50_trt_fp32.onnx"

    try:
        export_to_onnx(fp32_model, fp32_onnx)

        # ── Calibrator ─────────────────────────────────────────────────────────
        class CIFARCalibrator(trt.IInt8EntropyCalibrator2):
            """
            Feeds calibration batches to TensorRT.
            TRT uses these to compute per-tensor activation statistics,
            then solves: S* = argmin KL(P_fp32 || P_int8(·, S))
            """
            def __init__(self, loader, cache_file="trt_calib.cache"):
                super().__init__()
                self.loader     = iter(loader)
                self.batches    = 0
                self.cache_file = cache_file
                self.device_mem = None
                batch, _ = next(iter(loader))
                self.batch_size = batch.shape[0]
                self.buf_size   = int(np.prod(batch.shape) * 4)  # float32 bytes
                self.device_mem = cuda.mem_alloc(self.buf_size)

            def get_batch_size(self):
                return self.batch_size

            def get_batch(self, names):
                if self.batches >= CALIB_BATCHES:
                    return None
                try:
                    imgs, _ = next(self.loader)
                    cuda.memcpy_htod(self.device_mem, imgs.numpy().astype(np.float32))
                    self.batches += 1
                    return [int(self.device_mem)]
                except StopIteration:
                    return None

            def read_calibration_cache(self):
                if os.path.exists(self.cache_file):
                    with open(self.cache_file, "rb") as f:
                        return f.read()
                return None

            def write_calibration_cache(self, cache):
                with open(self.cache_file, "wb") as f:
                    f.write(cache)

        # ── Build engine ───────────────────────────────────────────────────────
        builder  = trt.Builder(TRT_LOGGER)
        network  = builder.create_network(
            1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
        )
        parser   = trt.OnnxParser(network, TRT_LOGGER)
        with open(fp32_onnx, "rb") as f:
            if not parser.parse(f.read()):
                raise RuntimeError(f"ONNX parse error: {parser.get_error(0)}")

        config = builder.create_builder_config()
        config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)  # 1 GB
        config.set_flag(trt.BuilderFlag.INT8)
        config.set_flag(trt.BuilderFlag.FP16)   # allow FP16 for unsupported INT8 ops

        calibrator = CIFARCalibrator(calib_loader)
        config.int8_calibrator = calibrator

        # Profile for dynamic batch
        profile = builder.create_optimization_profile()
        profile.set_shape("input",
                           min=(1, 3, 32, 32),
                           opt=(BATCH_SIZE, 3, 32, 32),
                           max=(BATCH_SIZE * 2, 3, 32, 32))
        config.add_optimization_profile(profile)

        print("        Building TRT INT8 engine (this may take several minutes)...")
        t_build = time.perf_counter()
        serialized = builder.build_serialized_network(network, config)
        build_time = time.perf_counter() - t_build
        print(f"        Engine built in {build_time:.1f}s")

        trt_path = "resnet50_int8.trt"
        with open(trt_path, "wb") as f:
            f.write(serialized)

        # ── Inference ─────────────────────────────────────────────────────────
        runtime = trt.Runtime(TRT_LOGGER)
        engine  = runtime.deserialize_cuda_engine(serialized)
        context = engine.create_execution_context()
        context.set_input_shape("input", (BATCH_SIZE, 3, 32, 32))

        # Allocate GPU buffers
        d_input  = cuda.mem_alloc(BATCH_SIZE * 3 * 32 * 32 * 4)
        d_output = cuda.mem_alloc(BATCH_SIZE * NUM_CLASSES * 4)
        stream   = cuda.Stream()

        def infer_trt(x_np):
            cuda.memcpy_htod_async(d_input, x_np.astype(np.float32), stream)
            context.execute_async_v2(
                bindings=[int(d_input), int(d_output)], stream_handle=stream.handle
            )
            out = np.empty((BATCH_SIZE, NUM_CLASSES), dtype=np.float32)
            cuda.memcpy_dtoh_async(out, d_output, stream)
            stream.synchronize()
            return out

        # Evaluate
        from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
        preds, labels = [], []
        for inputs, lbls in test_loader:
            out = infer_trt(inputs.numpy())
            preds.extend(np.argmax(out, 1))
            labels.extend(lbls.numpy())
        y_pred, y_true = np.array(preds), np.array(labels)
        metrics = {
            "accuracy" : float(accuracy_score(y_true, y_pred)),
            "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
            "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
            "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        }

        # GPU latency with CUDA events
        dummy_np = np.random.randn(BATCH_SIZE, 3, 32, 32).astype(np.float32)
        start_ev = cuda.Event()
        end_ev   = cuda.Event()
        for _ in range(5):
            infer_trt(dummy_np)
        timings = []
        for _ in range(INFERENCE_RUNS):
            start_ev.record()
            infer_trt(dummy_np)
            end_ev.record()
            end_ev.synchronize()
            timings.append(start_ev.time_till(end_ev))
        arr = np.array(timings)

        disk_mb = disk_size_mb(trt_path)

        row = {
            "backend"          : "tensorrt_int8",
            "description"      : (
                "TensorRT INT8 engine: ONNX → TRT with IInt8EntropyCalibrator2. "
                "Layer fusion (Conv-BN-ReLU), memory layout optimization, "
                "kernel autotuning at build time. "
                "S* = argmin_S KL(P_fp32 || P_int8(·, S)). "
                f"Engine built in {build_time:.1f}s; device-specific binary."
            ),
            "quantization_math": {
                "calibration"  : "KL-divergence minimization per tensor",
                "S"            : "S* = argmin_S KL(P_fp32(x) || P_int8(Q(x,S)))",
                "dtype_weights": "INT8 (per-channel symmetric)",
                "dtype_acts"   : "INT8 (per-tensor)",
                "fusion"       : "Conv+BN+ReLU fused to single kernel; residual add absorbed",
            },
            "build_time_sec"   : round(build_time, 2),
            "calibration_method": "Entropy (IInt8EntropyCalibrator2)",
            "calibration_samples": CALIB_SIZE,
            "accuracy"         : round(metrics["accuracy"],  6),
            "precision"        : round(metrics["precision"], 6),
            "recall"           : round(metrics["recall"],    6),
            "f1"               : round(metrics["f1"],        6),
            "accuracy_drop"    : round(baseline_acc - metrics["accuracy"], 6),
            "size_disk_mb"     : round(disk_mb, 4),
            "disk_saved_mb"    : round(base_disk - disk_mb, 4),
            "compression_ratio": round(base_disk / disk_mb, 4),
            "inference_latency": {
                "mean_ms"             : round(float(arr.mean()), 4),
                "std_ms"              : round(float(arr.std()),  4),
                "min_ms"              : round(float(arr.min()),  4),
                "max_ms"              : round(float(arr.max()),  4),
                "samples"             : INFERENCE_RUNS,
                "batch_size"          : BATCH_SIZE,
                "throughput_imgs_sec" : round(BATCH_SIZE / (float(arr.mean()) / 1000), 1),
                "timing_method"       : "CUDA events (GPU-synchronized)",
            },
            "flops": {
                "total_flops"           : flops_fp32.get("total_flops"),
                "gflops_fp32_equivalent": flops_fp32.get("gflops_fp32"),
                "gflops_int8_theoretical": flops_fp32.get("gflops_int8_theoretical"),
                "trt_fusion_note": (
                    "TRT fuses Conv-BN-ReLU into single op — memory ops reduced ~3×. "
                    "INT8 Tensor Core throughput: 4× FP32 on Ampere, 8× on Hopper."
                ),
            },
        }
        print(f"        → Acc: {metrics['accuracy']:.4f}  "
              f"Drop: {baseline_acc - metrics['accuracy']:+.4f}  "
              f"Disk: {disk_mb:.2f} MB  GPU: {float(arr.mean()):.1f} ms")

    except Exception as e:
        print(f"        → FAILED: {e}")
        row = {
            "backend"    : "tensorrt_int8",
            "description": f"FAILED: {e}",
            "accuracy"   : None,
        }
    finally:
        for f in [fp32_onnx, "trt_calib.cache"]:
            if os.path.exists(f):
                os.remove(f)

    return row


# ══════════════════════════════════════════════════════════════════════════════
# 3. PyTorch CUDA — Dynamic PTQ + PT2E Inductor INT8
# ══════════════════════════════════════════════════════════════════════════════
"""
PyTorch GPU quantization options:
  a) torch.quantization.quantize_dynamic — Linear → qint8
     Works on CUDA since PyTorch 2.0 for Linear layers only.
     Weights stored INT8, dequantized to FP16/FP32 for CUDA GEMM.
     Not true INT8 GEMM — dequant happens before GPU kernel.
     Still saves memory bandwidth (2× vs FP32).

  b) PT2E (PyTorch 2 Export) + X86InductorQuantizer → torch.compile
     torch.export captures the full ATen graph; quantizer annotates ops;
     prepare_pt2e inserts observers; calibrate; convert_pt2e freezes S/Z;
     torch.compile lowers to INT8 Triton/cuBLAS kernels via Inductor.
     Requires torch >= 2.3. True INT8 GEMM on CUDA.

PT2E import paths changed across PyTorch versions:
  torch < 2.4 : torch.ao.quantization.quantize_pt2e
                torch.ao.quantization.quantizer.x86_inductor_quantizer
                torch._export.capture_pre_autograd_graph
  torch >= 2.4: torch.ao.quantization.quantize_pt2e  (same)
                torch.ao.quantization.quantizer.xnnpack_quantizer (CPU)
                torch.export.export  (replaces capture_pre_autograd_graph)
  torch >= 2.5: torch.quantization.quantize_pt2e  (re-exported alias)

This function resolves all three variants at runtime.
"""

def _resolve_pt2e_imports():
    """
    Resolve PT2E symbols across PyTorch 2.3 / 2.4 / 2.5+ import paths.
    Returns (prepare_pt2e, convert_pt2e, Quantizer, get_config, export_fn)
    or raises ImportError with a clear message listing what was tried.

    Canonical locations by version
    ───────────────────────────────
    torch 2.3–2.4
      prepare/convert : torch.ao.quantization.quantize_pt2e
      quantizer       : torch.ao.quantization.quantizer.x86_inductor_quantizer
      export          : torch._export.capture_pre_autograd_graph
                        (deprecated in 2.4, removed in 2.5)

    torch 2.5+
      prepare/convert : torch.ao.quantization.quantize_pt2e  (still valid)
      quantizer       : torch.ao.quantization.quantizer.x86_inductor_quantizer
      export          : torch.export.export  (stable public API)
    """
    errors = []

    # ── Step 1: prepare_pt2e / convert_pt2e ───────────────────────────────────
    prepare_pt2e = convert_pt2e = None
    for mod_path in [
        "torch.ao.quantization.quantize_pt2e",   # 2.3–2.5+
        "torch.quantization.quantize_pt2e",       # re-export alias (some builds)
    ]:
        try:
            import importlib
            m = importlib.import_module(mod_path)
            prepare_pt2e = m.prepare_pt2e
            convert_pt2e = m.convert_pt2e
            break
        except (ImportError, AttributeError) as e:
            errors.append(f"  {mod_path}: {e}")

    if prepare_pt2e is None:
        raise ImportError(
            "Cannot find prepare_pt2e / convert_pt2e. Tried:\n" + "\n".join(errors) +
            "\nRequires PyTorch >= 2.3.  Install: pip install torch>=2.3"
        )

    # ── Step 2: X86InductorQuantizer ──────────────────────────────────────────
    Quantizer = get_config = None
    for mod_path, cls, cfg in [
        (
            "torch.ao.quantization.quantizer.x86_inductor_quantizer",
            "X86InductorQuantizer",
            "get_default_x86_inductor_quantization_config",
        ),
    ]:
        try:
            import importlib
            m = importlib.import_module(mod_path)
            Quantizer  = getattr(m, cls)
            get_config = getattr(m, cfg)
            break
        except (ImportError, AttributeError) as e:
            errors.append(f"  {mod_path}.{cls}: {e}")

    if Quantizer is None:
        raise ImportError(
            "Cannot find X86InductorQuantizer. Tried:\n" + "\n".join(errors) +
            "\nRequires PyTorch >= 2.3 built with Inductor."
        )

    # ── Step 3: graph export function ─────────────────────────────────────────
    # torch.export.export is the stable public API (2.1+).
    # capture_pre_autograd_graph was the older internal path (removed in 2.5).
    export_fn = None

    # Preferred: torch.export.export (wraps model + example_inputs)
    try:
        from torch.export import export as _torch_export

        def export_fn(model, example_args):
            # torch.export.export returns an ExportedProgram; PT2E needs the
            # .module() graph module for prepare_pt2e.
            ep = _torch_export(model, example_args)
            return ep.module()

    except (ImportError, AttributeError) as e:
        errors.append(f"  torch.export.export: {e}")

    # Fallback: capture_pre_autograd_graph (torch 2.3–2.4)
    if export_fn is None:
        try:
            from torch._export import capture_pre_autograd_graph as _cap

            def export_fn(model, example_args):
                return _cap(model, example_args)

        except (ImportError, AttributeError) as e:
            errors.append(f"  torch._export.capture_pre_autograd_graph: {e}")

    if export_fn is None:
        raise ImportError(
            "Cannot find a graph export function. Tried:\n" + "\n".join(errors)
        )

    return prepare_pt2e, convert_pt2e, Quantizer, get_config, export_fn


def run_torch_cuda_ptq(fp32_model, test_loader, calib_loader,
                        baseline_acc: float, base_disk: float,
                        device: str, flops_fp32: Dict) -> List[Dict]:
    print("\n  [3/3] PyTorch CUDA — Dynamic PTQ + PT2E Inductor INT8")
    rows = []
    import torch.nn as nn

    # ── 3a. Dynamic PTQ on CUDA ────────────────────────────────────────────────
    print("        3a. Dynamic PTQ (Linear → qint8) on CUDA")
    try:
        q_dyn = torch.quantization.quantize_dynamic(
            copy.deepcopy(fp32_model), {nn.Linear}, dtype=torch.qint8
        )
        # NOTE: Dynamic PTQ model keeps FP32 compute; weights are INT8 in memory.
        # CUDA does: dequant(INT8 weight) → FP16 → cuBLAS GEMM
        # True INT8 GEMM requires static calibration or TensorRT / PT2E.
        q_dyn = q_dyn.to(device).eval()

        metrics = evaluate_torch(q_dyn, test_loader, device)
        lat     = measure_gpu_throughput_ms(q_dyn, device)
        disk_mb = disk_size_mb(q_dyn.cpu())

        rows.append({
            "backend"         : "torch_cuda_dynamic",
            "description"     : (
                "PyTorch dynamic PTQ on CUDA: Linear weights → qint8 in memory. "
                "GPU execution: dequantizes weights to FP16 before cuBLAS GEMM. "
                "Not true INT8 GEMM — saves memory bandwidth, not compute. "
                "Conv2d stays FP32. No calibration needed."
            ),
            "quantization_math": {
                "scope"    : "Linear weights only",
                "storage"  : "qint8 (8-bit signed, 2× memory reduction)",
                "compute"  : "FP16 after dequant (not INT8 GEMM)",
                "S"        : "per-batch runtime from weight range",
                "gpu_note" : "Saves ~1.5-2× memory bandwidth; GPU compute same as FP16",
            },
            "ops_quantized"   : "Linear weights only (Conv2d stays FP32)",
            "calibration_samples": 0,
            "accuracy"        : round(metrics["accuracy"],  6),
            "precision"       : round(metrics["precision"], 6),
            "recall"          : round(metrics["recall"],    6),
            "f1"              : round(metrics["f1"],        6),
            "accuracy_drop"   : round(baseline_acc - metrics["accuracy"], 6),
            "size_disk_mb"    : round(disk_mb, 4),
            "disk_saved_mb"   : round(base_disk - disk_mb, 4),
            "compression_ratio": round(base_disk / disk_mb, 4) if disk_mb else None,
            "inference_latency": lat,
            "flops": {
                "total_flops"  : flops_fp32.get("total_flops"),
                "gflops"       : flops_fp32.get("gflops_fp32"),
                "effective_note": "FLOPs same as FP32; bandwidth saved ~2× for Linear",
            },
        })
        print(f"           → Acc: {metrics['accuracy']:.4f}  "
              f"GPU: {lat['mean_ms']:.1f} ms  Disk: {disk_mb:.2f} MB")
    except Exception as e:
        print(f"           → FAILED: {e}")
        rows.append({"backend": "torch_cuda_dynamic",
                     "description": f"FAILED: {e}", "accuracy": None})

    # ── 3b. PT2E + X86InductorQuantizer → torch.compile ──────────────────────
    print("        3b. PT2E (torch.export + X86InductorQuantizer) → torch.compile")
    try:
        # Resolve imports across PyTorch 2.3 / 2.4 / 2.5+
        prepare_pt2e, convert_pt2e, Quantizer, get_config, export_fn = (
            _resolve_pt2e_imports()
        )
        torch_ver = tuple(int(x) for x in torch.__version__.split(".")[:2])
        print(f"           PT2E imports resolved  "
              f"(torch {torch.__version__})")

        # Export ATen graph — works for both torch.export.export (2.5+) and
        # capture_pre_autograd_graph (2.3–2.4)
        example_args = (torch.randn(1, 3, 32, 32, device=device),)
        m = copy.deepcopy(fp32_model).to(device).eval()
        exported = export_fn(m, example_args)

        # Annotate + prepare observers
        quantizer = Quantizer()
        quantizer.set_global(get_config())
        prepared = prepare_pt2e(exported, quantizer)

        # Calibrate: run calib data so observers collect x_min, x_max
        # S = (x_max − x_min) / 255,  Z = round(−x_min / S)
        prepared.eval()
        with torch.no_grad():
            for i, (imgs, _) in enumerate(calib_loader):
                prepared(imgs.to(device))
                if i + 1 >= CALIB_BATCHES:
                    break

        # Freeze S/Z → INT8 ops (QLinearConv / QLinearMatMul in Inductor IR)
        q_pt2e = convert_pt2e(prepared)

        # Compile: Inductor lowers to INT8 Triton or cuBLAS kernels
        q_compiled = torch.compile(q_pt2e, mode="max-autotune",
                                    backend="inductor")

        metrics = evaluate_torch(q_compiled, test_loader, device)
        lat     = measure_gpu_throughput_ms(q_compiled, device)

        rows.append({
            "backend"         : "torch_pt2e_inductor_int8",
            "torch_version"   : torch.__version__,
            "description"     : (
                "PT2E (PyTorch 2 Export + X86InductorQuantizer): "
                "torch.export / capture_pre_autograd_graph → "
                "prepare_pt2e (inserts observers) → calibrate → "
                "convert_pt2e (freezes S/Z) → "
                "torch.compile(mode='max-autotune', backend='inductor'). "
                "Inductor lowers quantized ops to INT8 Triton/cuBLAS kernels. "
                "True INT8 GEMM on CUDA; Conv2d + Linear fully quantized."
            ),
            "quantization_math": {
                "S"        : "S = (x_max − x_min) / (2^8 − 1)  calibrated",
                "Z"        : "Z = round(−x_min / S)",
                "Q(x)"     : "Q(x) = round(x / S) + Z  → INT8 storage",
                "compute"  : "True INT8 GEMM via Inductor → Triton / cuBLAS INT8",
                "fusion"   : "Inductor fuses quantize + conv + dequantize into single kernel",
                "dtype_w"  : "INT8 symmetric per-channel",
                "dtype_act": "UINT8 per-tensor (affine)",
            },
            "ops_quantized"   : "Conv2d + Linear (all eligible ops in ATen graph)",
            "calibration_samples": CALIB_SIZE,
            "accuracy"        : round(metrics["accuracy"],  6),
            "precision"       : round(metrics["precision"], 6),
            "recall"          : round(metrics["recall"],    6),
            "f1"              : round(metrics["f1"],        6),
            "accuracy_drop"   : round(baseline_acc - metrics["accuracy"], 6),
            "inference_latency": lat,
            "flops": {
                "total_flops"           : flops_fp32.get("total_flops"),
                "gflops_fp32_equivalent": flops_fp32.get("gflops_fp32"),
                "gflops_int8_theoretical": flops_fp32.get("gflops_int8_theoretical"),
                "inductor_note": (
                    "Inductor fuses quant+conv+dequant into a single kernel, "
                    "so effective memory ops are fewer than the sequential count. "
                    "Actual throughput is measured by CUDA event wall-clock."
                ),
            },
        })
        print(f"           → Acc: {metrics['accuracy']:.4f}  "
              f"GPU: {lat['mean_ms']:.1f} ms")

    except ImportError as e:
        # PT2E not available on this PyTorch build — record skip, don't crash
        msg = str(e)
        print(f"           → SKIPPED (PT2E unavailable): {msg}")
        rows.append({
            "backend"    : "torch_pt2e_inductor_int8",
            "description": f"SKIPPED — PT2E not available on this PyTorch build: {msg}",
            "accuracy"   : None,
            "install_hint": "pip install torch>=2.3  (PT2E requires 2.3+)",
        })
    except Exception as e:
        print(f"           → FAILED: {e}")
        rows.append({"backend": "torch_pt2e_inductor_int8",
                     "description": f"FAILED: {e}", "accuracy": None})

    return rows


# ══════════════════════════════════════════════════════════════════════════════
# FP32 GPU Baseline
# ══════════════════════════════════════════════════════════════════════════════
def benchmark_fp32_gpu(fp32_model, test_loader, device, flops_fp32: Dict) -> Dict:
    """Measure FP32 GPU baseline for fair comparison."""
    print(f"\n  [0/3] FP32 GPU Baseline ({device})")
    model = copy.deepcopy(fp32_model).to(device).eval()
    metrics = evaluate_torch(model, test_loader, device)
    lat     = measure_gpu_throughput_ms(model, device)
    disk_mb = disk_size_mb(model.cpu())
    ram_mb  = sum(p.numel() for p in model.parameters()) * 4 / 1024 ** 2
    print(f"        Acc: {metrics['accuracy']:.4f}  "
          f"GPU: {lat['mean_ms']:.1f} ms  Disk: {disk_mb:.2f} MB")
    return {
        "accuracy"          : round(metrics["accuracy"],  6),
        "precision"         : round(metrics["precision"], 6),
        "recall"            : round(metrics["recall"],    6),
        "f1"                : round(metrics["f1"],        6),
        "size_disk_mb"      : round(disk_mb, 4),
        "ram_fp32_mb"       : round(ram_mb, 4),
        "inference_latency" : lat,
        "flops"             : flops_fp32,
        "device"            : str(device),
        "gpu_name"          : torch.cuda.get_device_name(device) if torch.cuda.is_available() else "N/A",
    }


# ══════════════════════════════════════════════════════════════════════════════
# Main
# ══════════════════════════════════════════════════════════════════════════════
def main():
    parser = argparse.ArgumentParser(description="GPU PTQ for ResNet-50/CIFAR-10")
    parser.add_argument("--install-deps", action="store_true",
                        help="Install required GPU packages and exit")
    parser.add_argument("--device", default="cuda",
                        help="Target device: cuda, cuda:0, cuda:1, cpu")
    args, _ = parser.parse_known_args()  # ignore Jupyter kernel args (--f=...)

    if args.install_deps:
        install_deps()
        return

    # ── Dependency check ───────────────────────────────────────────────────────
    missing = []
    if torch is None:
        missing.append("torch  →  pip install torch --index-url https://download.pytorch.org/whl/cu121")
    if onnx is None:
        missing.append("onnx   →  pip install onnx")
    if ort is None:
        missing.append("onnxruntime-gpu  →  pip install onnxruntime-gpu")
    if missing:
        print("\n⚠  Missing required packages:")
        for m in missing:
            print(f"   {m}")
        print("\nRun:  python gpu_ptq.py --install-deps")
        sys.exit(1)

    device = torch.device(args.device if torch.cuda.is_available() else "cpu")
    print(f"\n{'='*65}")
    print("  GPU Post-Training Quantization (PTQ) — ResNet-50 / CIFAR-10")
    print(f"  Device    : {device}")
    if torch.cuda.is_available():
        print(f"  GPU       : {torch.cuda.get_device_name(device)}")
        print(f"  CUDA      : {torch.version.cuda}")
        print(f"  cuDNN     : {torch.backends.cudnn.version()}")
    print("  Frameworks: ONNX Runtime GPU  |  TensorRT  |  PyTorch PT2E")
    print(f"{'='*65}")

    # ── Load baseline ──────────────────────────────────────────────────────────
    if not os.path.exists(BASELINE_MODEL_PATH):
        print(f"\n✗ Baseline model not found: {BASELINE_MODEL_PATH}")
        print("  Run the baseline training script first.")
        sys.exit(1)

    with open(BASELINE_METRICS_PATH) as f:
        baseline_metrics = json.load(f)
    baseline_acc = baseline_metrics["accuracy"]

    fp32_model   = load_baseline(BASELINE_MODEL_PATH)
    test_loader  = get_test_loader()
    calib_loader = get_calib_loader()

    # ── FLOPs (always computed on CPU model for consistency) ───────────────────
    print("\n  Computing FLOPs...")
    flops_fp32 = {}
    if fvcore is not None:
        try:
            flops_fp32 = count_flops_fvcore(fp32_model.cpu())
            print(f"  FLOPs (fvcore): {flops_fp32['gflops_fp32']:.3f} GFLOPs  "
                  f"|  INT8 theoretical: {flops_fp32['gflops_int8_theoretical']:.3f} GFLOPs")
        except Exception as e:
            print(f"  fvcore failed: {e}")

    if not flops_fp32:
        # Fallback to manual counting
        flops_fp32 = count_flops_manual(fp32_model.cpu())
        print(f"  FLOPs (manual): {flops_fp32['gflops']:.3f} GFLOPs")

    base_disk = disk_size_mb(fp32_model)

    # ── FP32 GPU Baseline ──────────────────────────────────────────────────────
    fp32_gpu_baseline = benchmark_fp32_gpu(fp32_model, test_loader, device, flops_fp32)

    # ── Run PTQ backends ───────────────────────────────────────────────────────
    all_results = []
    all_results.extend(
        run_onnx_ptq(fp32_model, test_loader, calib_loader,
                     baseline_acc, base_disk, device, flops_fp32)
    )
    all_results.append(
        run_tensorrt_ptq(fp32_model, test_loader, calib_loader,
                         baseline_acc, base_disk, device, flops_fp32)
    )
    all_results.extend(
        run_torch_cuda_ptq(fp32_model, test_loader, calib_loader,
                           baseline_acc, base_disk, device, flops_fp32)
    )

    # ── Best config ────────────────────────────────────────────────────────────
    valid = [r for r in all_results if r.get("accuracy") is not None]
    best  = min(valid, key=lambda r: (r.get("accuracy_drop", 99),
                                       r.get("size_disk_mb", 99))) if valid else {}

    # ── Build report ───────────────────────────────────────────────────────────
    report = {
        "experiment"  : "GPU Post-Training Quantization (PTQ)",
        "description" : (
            "GPU-native PTQ using three frameworks: "
            "(1) ONNX Runtime GPU with QDQ format and 3 calibration methods, "
            "(2) TensorRT INT8 engine with entropy calibration and layer fusion, "
            "(3) PyTorch CUDA: dynamic PTQ + PT2E Inductor INT8. "
            "FLOPs measured via fvcore (preferred) or manual hook-based counting."
        ),
        "ptq_math": {
            "quantize"    : "Q(x) = round(x / S) + Z",
            "dequantize"  : "x̂ = S · (Q(x) − Z)",
            "error"       : "ε = x − x̂,  |ε| ≤ S/2",
            "scale"       : "S = (x_max − x_min) / (2^b − 1)",
            "zero_point"  : "Z = round(−x_min / S)",
            "int8_levels" : "2^8 − 1 = 255",
            "gpu_advantage": (
                "INT8 Tensor Core throughput: 4× vs FP32 on Ampere (A100), "
                "8× on Hopper (H100). TensorRT fuses Conv-BN-ReLU into single kernel, "
                "eliminating intermediate memory round-trips."
            ),
        },
        "flops_methodology": {
            "tool"         : "fvcore.nn.FlopCountAnalysis (preferred) or manual hooks",
            "formula_conv" : "FLOPs = 2 × (Cin/groups) × Cout × Kh × Kw × Hout × Wout",
            "formula_linear": "FLOPs = 2 × in_features × out_features",
            "formula_bn"   : "FLOPs = 4 × C × H × W  (norm + scale + shift + var)",
            "macs_to_flops": "FLOPs = 2 × MACs  (one multiply + one accumulate per MAC)",
            "int8_note"    : (
                "Op count is identical for INT8 and FP32 models. "
                "Throughput gain = hardware parallelism: "
                "INT8 VNNI processes 4 elements per cycle vs 1 for FP32."
            ),
        },
        "gpu_info": {
            "device"    : str(device),
            "gpu_name"  : fp32_gpu_baseline.get("gpu_name", "N/A"),
            "cuda_ver"  : getattr(torch.version, "cuda", "N/A") if torch else "N/A",
            "cudnn_ver" : str(torch.backends.cudnn.version()) if torch and torch.cuda.is_available() else "N/A",
        },
        "baseline_fp32"     : baseline_metrics,
        "baseline_fp32_gpu" : fp32_gpu_baseline,
        "calibration_config": {
            "calib_size"   : CALIB_SIZE,
            "calib_batches": CALIB_BATCHES,
            "dataset"      : "CIFAR-10 training split (first 1024 images)",
        },
        "best_config" : {
            "backend"          : best.get("backend"),
            "calibration_method": best.get("calibration_method", best.get("observer")),
            "accuracy"         : best.get("accuracy"),
            "accuracy_drop"    : best.get("accuracy_drop"),
            "size_disk_mb"     : best.get("size_disk_mb"),
            "compression_ratio": best.get("compression_ratio"),
            "latency_mean_ms"  : best.get("inference_latency", {}).get("mean_ms"),
        } if best else {},
        "results"     : all_results,
        "framework_notes": {
            "onnxruntime_gpu": (
                "onnxruntime-gpu: requires onnxruntime-gpu (not onnxruntime). "
                "CUDAExecutionProvider uses cuDNN INT8 GEMM kernels. "
                "QDQ format inserts QuantizeLinear/DequantizeLinear pairs in graph. "
                "No engine compilation step → faster to deploy than TRT."
            ),
            "tensorrt": (
                "TensorRT: kernel autotuning at build time (GPU-specific binary). "
                "Requires pycuda for Python bindings. "
                "Highest throughput for production GPU serving. "
                "Engine is NOT portable across GPU architectures."
            ),
            "torch_pt2e": (
                "PT2E (PyTorch 2 Export): torch.export → quantizer annotations → "
                "prepare_pt2e → calibrate → convert_pt2e → torch.compile. "
                "Inductor backend generates INT8 Triton kernels. "
                "Most Pythonic; best integrated with PyTorch ecosystem."
            ),
            "why_not_cpu_static": (
                "torch.quantization static PTQ (prepare/convert) is CPU-only. "
                "For GPU INT8: use ORT-GPU, TensorRT, or PT2E+Inductor."
            ),
            "flops_vs_throughput": (
                "FLOPs count (compute ops) is the same for FP32 and INT8 models. "
                "Throughput gain from INT8 comes from hardware parallelism: "
                "INT8 SIMD/Tensor Core lanes process 4-8× more ops per clock. "
                "Memory bandwidth is also reduced ~4× (1 byte vs 4 bytes per weight)."
            ),
        },
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)

    print(f"\n{'='*65}")
    print(f"  ✓ Saved → {OUTPUT_JSON}")
    if best:
        print(f"  Best    : {best.get('backend')}  "
              f"acc={best.get('accuracy', 'N/A'):.4f}  "
              f"drop={best.get('accuracy_drop', 'N/A'):+.4f}")
        lat = best.get("inference_latency", {})
        if lat.get("mean_ms"):
            fp32_lat = fp32_gpu_baseline.get("inference_latency", {}).get("mean_ms", 1)
            speedup  = fp32_lat / lat["mean_ms"] if lat["mean_ms"] else 0
            print(f"  Speedup : {speedup:.2f}× vs FP32 GPU  "
                  f"({lat['mean_ms']:.1f} ms vs {fp32_lat:.1f} ms)")
    print(f"{'='*65}\n")


if __name__ == "__main__":
    main()


  GPU Post-Training Quantization (PTQ) — ResNet-50 / CIFAR-10
  Device    : cuda
  GPU       : NVIDIA GeForce RTX 5070 Laptop GPU
  CUDA      : 12.8
  cuDNN     : 92000
  Frameworks: ONNX Runtime GPU  |  TensorRT  |  PyTorch PT2E

  Computing FLOPs...
  FLOPs (manual): 2.609 GFLOPs

  [0/3] FP32 GPU Baseline (cuda)


W0429 00:11:33.562000 20372 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


        Acc: 0.9320  GPU: 51.3 ms  Disk: 90.05 MB

  [1/3] ONNX Runtime — Static INT8 PTQ
        Framework: onnxruntime-gpu + cuDNN INT8 kernels
        Ops:       QLinearConv, QLinearMatMul, QLinearAdd
        S/Z:       calibrated once from calib data; frozen in graph
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "e:\baseline_resnet50_cifar10\env\Lib\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "e:\baseline_resnet50_cifar10\env\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "e:\baseline_resnet50_cifar10\env\Lib\site-packages\onnxscript\version_converter\__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "e:\baseline_resnet50_cifar10\env\Lib\site-packages\onnx\version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
RuntimeError: D:\a\onnx\onnx\onnx/version

[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
        ✓ ONNX exported → resnet50_fp32.onnx  (0.22 MB, opset=17)
        Calibration: MinMax      

 → Acc: 0.9319  Drop: +0.0001  Disk: 90.18 MB  Lat: 63.5 ms (CUDAExecutionProvider)
        Calibration: Entropy     

 → FAILED: [ONNXRuntimeError] : 6 : RUNTIME_EXCEPTION : Non-zero status code returned while running Conv node. Name:'node_Conv_770' Status Message: E:\_work\1\s\onnxruntime\core\framework\bfc_arena.cc:358 onnxruntime::BFCArena::AllocateRawInternal Failed to allocate memory for requested buffer of size 33554432

        Calibration: Percentile  

 → FAILED: [ONNXRuntimeError] : 6 : RUNTIME_EXCEPTION : Non-zero status code returned while running Conv node. Name:'node_Conv_770' Status Message: bad allocation

  [2/3] TensorRT — INT8 PTQ
        ⚠ TensorRT/pycuda not available: No module named 'tensorrt'
          Install: pip install tensorrt pycuda

  [3/3] PyTorch CUDA — Dynamic PTQ + PT2E Inductor INT8
        3a. Dynamic PTQ (Linear → qint8) on CUDA
           → FAILED: Could not run 'quantized::linear_dynamic' with arguments from the 'CUDA' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'quantized::linear_dynamic' is only available for these backends: [CPU, Meta, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Named, Conjugate, Negative, ZeroTensor, ADInplaceOrView, AutogradO